# Mario RAM RL: World 5-2 Mastery Colab

This notebook runs the updated World 5-2 training pipeline:

- RAM observations
- RecurrentPPO with `MlpLstmPolicy`
- `mario-secrets` action space
- `stage-score` reward
- single-life and single-stage episodes
- VecNormalize
- completion-aware evaluation and video export

Before running the serious training cell, make sure your Drive has:

```text
MyDrive/mario_rl/roms/<your Super Mario Bros .nes file>
MyDrive/mario_rl/custom_integrations/SuperMarioBros-Nes-v0/Level5-2.state
```

## 1. Runtime

Use **Runtime > Change runtime type** and choose a GPU if available. The emulator workers are CPU-heavy, so `--device cpu` is still the default for RAM + MLP/LSTM training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/hbofz/mario-rl-ram.git'
REPO_DIR = Path('/content/mario-rl-ram')
DRIVE_ROOT = Path('/content/drive/MyDrive/mario_rl')
ROM_DIR = DRIVE_ROOT / 'roms'
DRIVE_STATE = DRIVE_ROOT / 'custom_integrations/SuperMarioBros-Nes-v0/Level5-2.state'
REPO_STATE = REPO_DIR / 'custom_integrations/SuperMarioBros-Nes-v0/Level5-2.state'
MODEL_DIR = DRIVE_ROOT / 'models'
LOG_DIR = DRIVE_ROOT / 'runs'
VIDEO_DIR = DRIVE_ROOT / 'videos/world-5-2'

RUN_NAME = 'recurrent-ppo-ram-5-2-gpu-fast'
TIMESTEPS = 10_000_000
MORE_TIMESTEPS = 5_000_000
CHECKPOINT_FREQ = 250_000
EVAL_FREQ = 500_000
EVAL_EPISODES = 1

for path in [DRIVE_ROOT, ROM_DIR, MODEL_DIR, LOG_DIR, VIDEO_DIR, DRIVE_STATE.parent]:
    path.mkdir(parents=True, exist_ok=True)

print('ROM_DIR:', ROM_DIR)
print('DRIVE_STATE:', DRIVE_STATE)
print('MODEL_SAVE_DIR:', MODEL_DIR / RUN_NAME)
print('LOG_SAVE_DIR:', LOG_DIR / RUN_NAME)
print('CHECKPOINT_FREQ:', CHECKPOINT_FREQ)
print('EVAL_FREQ:', EVAL_FREQ)
print('EVAL_EPISODES:', EVAL_EPISODES)

## 2. Clone And Install

This clones the repo fresh if needed, then installs the local package. If you are testing an unmerged branch, edit `REPO_URL` above or add a checkout command after clone.

In [ ]:
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
!pip install -e .

## 2.5. GPU Check

This run is configured for RecurrentPPO on GPU. If this cell fails, switch Colab to a GPU runtime before training.

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available. In Colab, use Runtime > Change runtime type > GPU.')

print('GPU:', torch.cuda.get_device_name(0))
torch.set_float32_matmul_precision('high')
print('matmul precision:', torch.get_float32_matmul_precision())

## 3. Import The ROM

Upload your legally obtained ROM to `MyDrive/mario_rl/roms/` first. This imports it into Stable-Retro inside the Colab runtime.

In [ ]:
roms = sorted(ROM_DIR.glob('*.nes'))
if not roms:
    print(f'No .nes ROM found in {ROM_DIR}. Uploading through Colab now...')
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        if name.lower().endswith('.nes'):
            target = ROM_DIR / name
            target.write_bytes(data)
            print('Saved ROM to', target)
    roms = sorted(ROM_DIR.glob('*.nes'))

if not roms:
    raise FileNotFoundError(f'No .nes ROM found in {ROM_DIR}.')

print('Found ROMs:')
for rom in roms:
    print(' -', rom)

!python -m stable_retro.import {ROM_DIR}

## 4. Copy The World 5-2 Savestate

The ROM has World 5-2, but RL needs a Stable-Retro `.state` snapshot to reset directly there. Put your captured state at:

```text
MyDrive/mario_rl/custom_integrations/SuperMarioBros-Nes-v0/Level5-2.state
```

This cell copies it into the repo-local custom integration folder used by `mario-train`.

In [ ]:
import shutil

REPO_STATE.parent.mkdir(parents=True, exist_ok=True)
if not DRIVE_STATE.exists() and not REPO_STATE.exists():
    print('No Level5-2.state found in Drive or repo. Uploading through Colab now...')
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        if name == 'Level5-2.state' or name.endswith('.state'):
            DRIVE_STATE.write_bytes(data)
            print('Saved state to Drive:', DRIVE_STATE)
            break

if DRIVE_STATE.exists():
    shutil.copy2(DRIVE_STATE, REPO_STATE)
    print('Copied Level5-2.state from Drive to repo:', REPO_STATE)
elif REPO_STATE.exists():
    print('Using Level5-2.state already present in repo:', REPO_STATE)
else:
    raise FileNotFoundError(
        'Missing Level5-2.state. Put it at '
        f'{DRIVE_STATE} or commit it under {REPO_STATE}.'
    )

!ls -lh {REPO_STATE.parent}

## 5. Environment Checks

This confirms Stable-Retro sees the ROM and that the 5-2 state starts at displayed World 5-2 (`levelHi=4`, `levelLo=1`).

In [ ]:
!mario-doctor

!mario-smoke \
  --state Level5-2 \
  --custom-integration-path custom_integrations \
  --expected-stage 5-2 \
  --reward-mode stage-score \
  --action-mode mario-secrets \
  --single-stage \
  --steps 300

## 6. Tiny Wiring Run

Run this first after setup. It is intentionally tiny; it only proves training can initialize on 5-2 before starting a long Colab job.

In [ ]:
!mario-train \
  --timesteps 1024 \
  --n-envs 1 \
  --n-steps 128 \
  --batch-size 128 \
  --eval-freq 0 \
  --state Level5-2 \
  --custom-integration-path custom_integrations \
  --expected-stage 5-2 \
  --action-mode mario-secrets \
  --reward-mode stage-score \
  --single-stage \
  --single-life \
  --no-auto-resume \
  --no-vecnormalize \
  --run-name verify-level5-2-colab \
  --model-dir {MODEL_DIR} \
  --log-dir {LOG_DIR} \
  --device cpu

## 7. Main World 5-2 Training

This is the GPU-fast RecurrentPPO command. It saves models and checkpoints to Drive under:

```text
/content/drive/MyDrive/mario_rl/models/recurrent-ppo-ram-5-2-gpu-fast/
/content/drive/MyDrive/mario_rl/runs/recurrent-ppo-ram-5-2-gpu-fast/
```

Checkpoints save every `250,000` timesteps by default here. Eval runs every `500,000` timesteps with 1 episode to avoid crushing FPS while still giving progress signals.

In [ ]:
!mario-train \
  --algo recurrent-ppo \
  --timesteps {TIMESTEPS} \
  --n-envs 16 \
  --n-steps 512 \
  --batch-size 2048 \
  --n-epochs 1 \
  --gamma 0.995 \
  --gae-lambda 0.95 \
  --ent-coef 0.01 \
  --state Level5-2 \
  --custom-integration-path custom_integrations \
  --expected-stage 5-2 \
  --action-mode mario-secrets \
  --reward-mode stage-score \
  --single-stage \
  --single-life \
  --vecnormalize \
  --checkpoint-freq {CHECKPOINT_FREQ} \
  --eval-freq {EVAL_FREQ} \
  --eval-episodes {EVAL_EPISODES} \
  --run-name {RUN_NAME} \
  --model-dir {MODEL_DIR} \
  --log-dir {LOG_DIR} \
  --device cuda

## 8. Resume Or Extend Training

The trainer auto-resumes from the latest checkpoint in the Drive model folder for `RUN_NAME`. This cell repeats the GPU-fast settings so an extension run does not accidentally fall back to slower defaults.

In [ ]:
!mario-train \
  --algo recurrent-ppo \
  --timesteps {MORE_TIMESTEPS} \
  --n-envs 16 \
  --n-steps 512 \
  --batch-size 2048 \
  --n-epochs 1 \
  --state Level5-2 \
  --custom-integration-path custom_integrations \
  --expected-stage 5-2 \
  --action-mode mario-secrets \
  --reward-mode stage-score \
  --single-stage \
  --single-life \
  --vecnormalize \
  --checkpoint-freq {CHECKPOINT_FREQ} \
  --eval-freq {EVAL_FREQ} \
  --eval-episodes {EVAL_EPISODES} \
  --run-name {RUN_NAME} \
  --model-dir {MODEL_DIR} \
  --log-dir {LOG_DIR} \
  --device cuda

## 9. Evaluate Best And Final Models

Use `best_stage_score/best_model.zip` when you care about completion rate first and score second. Use `final_model.zip` to inspect the last checkpoint.

In [ ]:
BEST_MODEL = MODEL_DIR / RUN_NAME / 'best_stage_score/best_model.zip'
BEST_VECNORMALIZE = MODEL_DIR / RUN_NAME / 'best_stage_score/best_vecnormalize.pkl'
FINAL_MODEL = MODEL_DIR / RUN_NAME / 'final_model.zip'
FINAL_VECNORMALIZE = MODEL_DIR / RUN_NAME / 'vecnormalize.pkl'

print('BEST_MODEL:', BEST_MODEL, BEST_MODEL.exists())
print('FINAL_MODEL:', FINAL_MODEL, FINAL_MODEL.exists())

In [ ]:
if BEST_MODEL.exists() and BEST_VECNORMALIZE.exists():
    !mario-eval \
      --model {BEST_MODEL} \
      --vecnormalize {BEST_VECNORMALIZE} \
      --episodes 5 \
      --state Level5-2 \
      --custom-integration-path custom_integrations \
      --expected-stage 5-2 \
      --action-mode mario-secrets \
      --reward-mode stage-score \
      --single-stage \
      --video-dir {VIDEO_DIR / 'best-deterministic'} \
      --device cpu
else:
    print('Best stage-score model not found yet. Run training/eval checkpoints first.')

In [ ]:
if FINAL_MODEL.exists() and FINAL_VECNORMALIZE.exists():
    !mario-eval \
      --model {FINAL_MODEL} \
      --vecnormalize {FINAL_VECNORMALIZE} \
      --episodes 5 \
      --state Level5-2 \
      --custom-integration-path custom_integrations \
      --expected-stage 5-2 \
      --action-mode mario-secrets \
      --reward-mode stage-score \
      --single-stage \
      --video-dir {VIDEO_DIR / 'final-deterministic'} \
      --device cpu
else:
    print('Final model not found yet. Run training first.')

## 10. TensorBoard

Look for `mario/*`, `eval/*`, and `eval_stage_score/*` metrics. The key one is `eval_stage_score/completion_rate`; score only matters after the model can clear 5-2.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}